In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Step 38 Source CSV configuration
base_path = "Files/landing"

source_paths = {
    "customers": f"{base_path}/customers/customers.csv",
    "products": f"{base_path}/products/products.csv",
    "sales": f"{base_path}/sales/sales.csv"
}

print("Production Bronze notebook initialized")

# Validate source file accessibility
for name, path in source_paths.items():
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
    )

    print(f"{name}: {df.count()} records")


In [ ]:

# STEP 48: Source Fingerprint Baseline

import hashlib

def calculate_file_hash(path):
    """
    Calculate SHA-256 hash of source file bytes.
    """
    file_df = (
        spark.read
        .format("binaryFile")
        .load(path)
    )

    rows = file_df.select("content").collect()

    if len(rows) != 1:
        raise ValueError(
            f"Expected one file at {path}, found {len(rows)}"
        )

    return hashlib.sha256(
        bytes(rows[0]["content"])
    ).hexdigest()

source_hashes = {}

for name, path in source_paths.items():
    file_hash = calculate_file_hash(path)
    source_hashes[name] = file_hash

    print(f"{name}: {file_hash}")

print("Source fingerprint calculation completed")


In [ ]:

# STEP 49: Initialize Source Fingerprint Control Table

from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType
)

control_table = "ctl_source_file_hashes"

control_schema = StructType([
    StructField("source_name", StringType(), False),
    StructField("file_hash", StringType(), False),
    StructField("processed_at", TimestampType(), True)
])

# Create an empty control table only if it doesn't exist
if not spark.catalog.tableExists(control_table):

    empty_control_df = spark.createDataFrame(
        [],
        schema=control_schema
    )

    (
        empty_control_df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(control_table)
    )

    print("Source fingerprint control table created")

else:
    print("Source fingerprint control table already exists")

# Verify control table
control_df = spark.table(control_table)

print("Control Table Records:", control_df.count())

display(control_df)


In [ ]:

# STEP 65: Build Safe Incremental Processing Plan

# Read previously registered fingerprints
stored_hashes = {
    row["source_name"]: row["file_hash"]
    for row in spark.table("ctl_source_file_hashes")
        .select("source_name", "file_hash")
        .collect()
}

# Detect source changes
source_changes = {}

for source_name, source_path in source_paths.items():

    current_hash = calculate_file_hash(source_path)

    source_changes[source_name] = (
        current_hash != stored_hashes.get(source_name)
    )

# Check whether target tables exist
customers_missing = not spark.catalog.tableExists(
    "bronze_customers_incremental"
)

products_missing = not spark.catalog.tableExists(
    "bronze_products_incremental"
)

sales_missing = not spark.catalog.tableExists(
    "bronze_sales_incremental"
)

quarantine_missing = not spark.catalog.tableExists(
    "bronze_sales_quarantine"
)

# Build processing plan
process_customers = (
    source_changes["customers"] or customers_missing
)

process_products = (
    source_changes["products"] or products_missing
)

process_sales = (
    source_changes["sales"]
    or process_customers
    or process_products
    or sales_missing
    or quarantine_missing
)

print("SAFE INCREMENTAL PROCESSING PLAN")
print("--------------------------------")
print("Process Customers:", process_customers)
print("Process Products:", process_products)
print("Process Sales:", process_sales)

if not any([
    process_customers,
    process_products,
    process_sales
]):
    print("No processing required")
else:
    print("Incremental processing required")


In [ ]:

# STEP 77: Customers MERGE with Soft Delete Support
# Replaces original Step 55

from delta.tables import DeltaTable
from pyspark.sql import functions as F

customer_table = "bronze_customers_incremental"

should_process_customers = (
    process_customers
    or not spark.catalog.tableExists(customer_table)
)

if should_process_customers:

    incoming_customers = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_paths["customers"])
    )

    # Validate primary key
    if incoming_customers.filter(
        F.col("customer_id").isNull()
    ).count() > 0:
        raise ValueError("Null customer IDs found")

    if incoming_customers.groupBy("customer_id") \
        .count().filter("count > 1").count() > 0:
        raise ValueError("Duplicate customer IDs found")

    incoming_customers = (
        incoming_customers
        .withColumn("is_deleted", F.lit(False))
        .withColumn(
            "deleted_at",
            F.lit(None).cast("timestamp")
        )
    )

    if not spark.catalog.tableExists(customer_table):

        incoming_customers.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(customer_table)

    else:

        target = DeltaTable.forName(
            spark, customer_table
        )

        (
            target.alias("target")
            .merge(
                incoming_customers.alias("source"),
                "target.customer_id = source.customer_id"
            )
            .whenMatchedUpdate(
                condition="""
                    NOT (target.customer_name <=> source.customer_name)
                    OR NOT (target.city <=> source.city)
                    OR NOT (target.signup_date <=> source.signup_date)
                    OR target.is_deleted IS NULL
                    OR target.is_deleted = true
                """,
                set={
                    "customer_name": "source.customer_name",
                    "city": "source.city",
                    "signup_date": "source.signup_date",
                    "is_deleted": "false",
                    "deleted_at": "NULL"
                }
            )
            .whenNotMatchedInsertAll()
            .whenNotMatchedBySourceUpdate(
                condition="target.is_deleted = false",
                set={
                    "is_deleted": "true",
                    "deleted_at": "current_timestamp()"
                }
            )
            .execute()
        )

    print("Customers MERGE completed with soft deletes")

else:
    print("Customers unchanged — skipped")

customers_df = spark.table(customer_table)

print("Total Customers:", customers_df.count())
print(
    "Active Customers:",
    customers_df.filter("is_deleted = false").count()
)
print(
    "Deleted Customers:",
    customers_df.filter("is_deleted = true").count()
)


In [ ]:

# STEP 78: Products MERGE with Soft Delete Support
# Replaces original Step 56

from delta.tables import DeltaTable
from pyspark.sql import functions as F

product_table = "bronze_products_incremental"

should_process_products = (
    process_products
    or not spark.catalog.tableExists(product_table)
)

if should_process_products:

    incoming_products = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_paths["products"])
    )

    # Validate product IDs
    if incoming_products.filter(
        F.col("product_id").isNull()
    ).count() > 0:
        raise ValueError("Null product IDs found")

    if incoming_products.groupBy("product_id") \
        .count().filter("count > 1").count() > 0:
        raise ValueError("Duplicate product IDs found")

    incoming_products = (
        incoming_products
        .withColumn("is_deleted", F.lit(False))
        .withColumn(
            "deleted_at",
            F.lit(None).cast("timestamp")
        )
    )

    if not spark.catalog.tableExists(product_table):

        incoming_products.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(product_table)

    else:

        target = DeltaTable.forName(
            spark, product_table
        )

        (
            target.alias("target")
            .merge(
                incoming_products.alias("source"),
                "target.product_id = source.product_id"
            )
            .whenMatchedUpdate(
                condition="""
                    NOT (target.product_name <=> source.product_name)
                    OR NOT (target.category <=> source.category)
                    OR NOT (target.unit_price <=> source.unit_price)
                    OR target.is_deleted IS NULL
                    OR target.is_deleted = true
                """,
                set={
                    "product_name": "source.product_name",
                    "category": "source.category",
                    "unit_price": "source.unit_price",
                    "is_deleted": "false",
                    "deleted_at": "NULL"
                }
            )
            .whenNotMatchedInsertAll()
            .whenNotMatchedBySourceUpdate(
                condition="target.is_deleted = false",
                set={
                    "is_deleted": "true",
                    "deleted_at": "current_timestamp()"
                }
            )
            .execute()
        )

    print("Products MERGE completed with soft deletes")

else:
    print("Products unchanged — skipped")

products_df = spark.table(product_table)

print("Total Products:", products_df.count())
print(
    "Active Products:",
    products_df.filter("is_deleted = false").count()
)
print(
    "Deleted Products:",
    products_df.filter("is_deleted = true").count()
)


In [ ]:

# STEP 57: Conditional Sales Source Deduplication

from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Process Sales when source/dimensions change
# or either target table is missing
should_process_sales = (
    process_sales
    or not spark.catalog.tableExists("bronze_sales_incremental")
    or not spark.catalog.tableExists("bronze_sales_quarantine")
)

if should_process_sales:

    print("Sales processing started")

    # Read Sales CSV
    incoming_sales_raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_paths["sales"])
    )

    # Validate null Sale IDs
    null_ids = incoming_sales_raw.filter(
        F.col("sale_id").isNull()
    ).count()

    if null_ids > 0:
        raise ValueError(
            f"Sales source contains {null_ids} null Sale IDs"
        )

    # Keep the latest sale_date for duplicate Sale IDs
    sales_window = (
        Window.partitionBy("sale_id")
        .orderBy(F.col("sale_date").desc())
    )

    incoming_sales_clean = (
        incoming_sales_raw
        .withColumn(
            "row_num",
            F.row_number().over(sales_window)
        )
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    # Validate deduplication
    raw_count = incoming_sales_raw.count()
    clean_count = incoming_sales_clean.count()

    duplicate_count = (
        clean_count
        - incoming_sales_clean.select("sale_id").distinct().count()
    )

    print("Raw Sales:", raw_count)
    print("Deduplicated Sales:", clean_count)
    print("Duplicate IDs After Cleaning:", duplicate_count)

    print("Retained S0050 Record:")

    display(
        incoming_sales_clean.filter(
            F.col("sale_id") == "S0050"
        )
    )

else:

    print("Sales source unchanged — skipping deduplication")


In [ ]:

# STEP 79: Sales Referential Integrity with Soft Deletes
# Replaces original Step 58

from pyspark.sql import functions as F

if should_process_sales:

    print("Sales referential integrity validation started")

    # Read only active Customers and Products
    customers = (
        spark.table("bronze_customers_incremental")
        .filter(F.col("is_deleted") == False)
    )

    products = (
        spark.table("bronze_products_incremental")
        .filter(F.col("is_deleted") == False)
    )

    # Sales with active Customer and Product references
    valid_sales = (
        incoming_sales_clean
        .join(
            customers.select("customer_id"),
            "customer_id",
            "left_semi"
        )
        .join(
            products.select("product_id"),
            "product_id",
            "left_semi"
        )
    )

    # Sales referencing missing or inactive dimensions
    invalid_sales = (
        incoming_sales_clean
        .join(
            valid_sales.select("sale_id"),
            "sale_id",
            "left_anti"
        )
    )

    # Validate classification counts
    total_sales = incoming_sales_clean.count()
    valid_count = valid_sales.count()
    invalid_count = invalid_sales.count()

    print("Deduplicated Sales:", total_sales)
    print("Valid Sales:", valid_count)
    print("Invalid Sales:", invalid_count)

    if total_sales != valid_count + invalid_count:
        raise ValueError(
            "Sales classification count mismatch"
        )

    print("Invalid Sales Details:")

    invalid_sales.select(
        "sale_id",
        "customer_id",
        "product_id"
    ).show(truncate=False)

else:

    print(
        "Sales source unchanged — "
        "skipping referential integrity validation"
    )


In [ ]:

# STEP 59: Conditional Production Sales MERGE

from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "bronze_sales_incremental"

if should_process_sales:

    print("Sales processing started")

    incoming_sales = valid_sales

    # Validate incoming Sale IDs
    total = incoming_sales.count()

    unique = (
        incoming_sales
        .select("sale_id")
        .distinct()
        .count()
    )

    null_keys = incoming_sales.filter(
        F.col("sale_id").isNull()
    ).count()

    if total != unique or null_keys > 0:
        raise ValueError("Invalid Sale IDs detected")

    # Initial load or MERGE
    if not spark.catalog.tableExists(target_table):

        (
            incoming_sales.write
            .format("delta")
            .mode("errorifexists")
            .saveAsTable(target_table)
        )

        print("Sales Delta table initialized")

    else:

        target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target.alias("target")
            .merge(
                incoming_sales.alias("source"),
                "target.sale_id = source.sale_id"
            )
            .whenMatchedUpdate(
                condition="""
                    NOT (
                        target.customer_id <=> source.customer_id
                        AND target.product_id <=> source.product_id
                        AND target.sale_date <=> source.sale_date
                        AND target.quantity <=> source.quantity
                        AND target.unit_price <=> source.unit_price
                    )
                """,
                set={
                    "customer_id": "source.customer_id",
                    "product_id": "source.product_id",
                    "sale_date": "source.sale_date",
                    "quantity": "source.quantity",
                    "unit_price": "source.unit_price"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("Sales Delta MERGE completed")

else:

    print("Sales source unchanged — skipping MERGE")

# Validate target in both cases
result = spark.table(target_table)

total_sales = result.count()
unique_sales = result.select("sale_id").distinct().count()

print("Total Sales:", total_sales)
print("Unique Sale IDs:", unique_sales)

print(
    "Invalid S0052 Records:",
    result.filter(
        F.col("sale_id") == "S0052"
    ).count()
)

if total_sales != unique_sales:
    raise ValueError("Duplicate Sale IDs in target table")

if result.filter(F.col("sale_id").isNull()).count() > 0:
    raise ValueError("Null Sale IDs in target table")


In [ ]:

# STEP 80: Conditional Sales Quarantine MERGE
# Supports missing and soft-deleted dimensions
# Replaces original Step 60

from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "bronze_sales_quarantine"

if should_process_sales:

    print("Sales quarantine processing started")

    # Read all dimension records, including soft-deleted ones
    all_customers = spark.table("bronze_customers_incremental")
    all_products = spark.table("bronze_products_incremental")

    # Classify invalid sales by dimension status
    customer_status = (
        all_customers
        .select(
            "customer_id",
            F.col("is_deleted").alias("customer_is_deleted"),
            F.lit(True).alias("customer_exists")
        )
    )

    product_status = (
        all_products
        .select(
            "product_id",
            F.col("is_deleted").alias("product_is_deleted"),
            F.lit(True).alias("product_exists")
        )
    )

    classified = (
        invalid_sales
        .join(customer_status, "customer_id", "left")
        .join(product_status, "product_id", "left")
    )

    customer_reason = (
        F.when(
            F.col("customer_exists").isNull(),
            F.lit("CUSTOMER_NOT_FOUND")
        )
        .when(
            F.col("customer_is_deleted") == True,
            F.lit("CUSTOMER_SOFT_DELETED")
        )
    )

    product_reason = (
        F.when(
            F.col("product_exists").isNull(),
            F.lit("PRODUCT_NOT_FOUND")
        )
        .when(
            F.col("product_is_deleted") == True,
            F.lit("PRODUCT_SOFT_DELETED")
        )
    )

    # Combine reasons when both dimensions are invalid
    incoming_quarantine = (
        classified
        .withColumn("customer_reason", customer_reason)
        .withColumn("product_reason", product_reason)
        .withColumn(
            "rejection_reason",
            F.when(
                F.col("customer_reason").isNotNull()
                & F.col("product_reason").isNotNull(),
                F.concat_ws(
                    "_AND_",
                    F.col("customer_reason"),
                    F.col("product_reason")
                )
            )
            .otherwise(
                F.coalesce(
                    F.col("customer_reason"),
                    F.col("product_reason")
                )
            )
        )
        .drop(
            "customer_exists",
            "customer_is_deleted",
            "product_exists",
            "product_is_deleted",
            "customer_reason",
            "product_reason"
        )
    )

    # Validate incoming quarantine records
    total = incoming_quarantine.count()
    unique = incoming_quarantine.select("sale_id").distinct().count()
    null_keys = incoming_quarantine.filter(
        F.col("sale_id").isNull()
    ).count()

    null_reasons = incoming_quarantine.filter(
        F.col("rejection_reason").isNull()
    ).count()

    if total != unique or null_keys > 0 or null_reasons > 0:
        raise ValueError(
            "Invalid quarantine source: duplicate IDs, "
            "null IDs, or missing rejection reasons"
        )

    # Initialize or MERGE quarantine table
    if not spark.catalog.tableExists(target_table):

        (
            incoming_quarantine.write
            .format("delta")
            .mode("errorifexists")
            .saveAsTable(target_table)
        )

        print("Sales Quarantine Delta table initialized")

    else:

        target = DeltaTable.forName(spark, target_table)

        (
            target.alias("target")
            .merge(
                incoming_quarantine.alias("source"),
                "target.sale_id = source.sale_id"
            )
            .whenMatchedUpdate(
                condition="""
                    NOT (
                        target.customer_id <=> source.customer_id
                        AND target.product_id <=> source.product_id
                        AND target.sale_date <=> source.sale_date
                        AND target.quantity <=> source.quantity
                        AND target.unit_price <=> source.unit_price
                        AND target.rejection_reason <=> source.rejection_reason
                    )
                """,
                set={
                    "customer_id": "source.customer_id",
                    "product_id": "source.product_id",
                    "sale_date": "source.sale_date",
                    "quantity": "source.quantity",
                    "unit_price": "source.unit_price",
                    "rejection_reason": "source.rejection_reason"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("Sales Quarantine MERGE completed")

else:

    print("Sales source unchanged — skipping quarantine MERGE")

# Validate existing quarantine table
result = spark.table(target_table)

total_quarantined = result.count()
unique_quarantined = result.select("sale_id").distinct().count()

print("Quarantined Sales:", total_quarantined)
print("Unique Quarantined Sale IDs:", unique_quarantined)

if total_quarantined != unique_quarantined:
    raise ValueError("Duplicate Sale IDs in quarantine table")

if result.filter(F.col("sale_id").isNull()).count() > 0:
    raise ValueError("Null Sale IDs in quarantine table")

display(
    result.select(
        "sale_id",
        "customer_id",
        "rejection_reason"
    )
)


In [ ]:

# STEP 61: Conditional Sales and Quarantine Reconciliation

from delta.tables import DeltaTable

if should_process_sales:

    print("Sales reconciliation started")

    valid_snapshot = valid_sales.cache()
    invalid_snapshot = incoming_quarantine.cache()

    try:
        valid_count = valid_snapshot.count()
        invalid_count = invalid_snapshot.count()

        overlap = (
            valid_snapshot.select("sale_id")
            .intersect(invalid_snapshot.select("sale_id"))
            .count()
        )

        if overlap > 0:
            raise ValueError(
                "Sales classification overlap detected"
            )

        # Remove newly invalid sales from active Sales
        sales_target = DeltaTable.forName(
            spark,
            "bronze_sales_incremental"
        )

        (
            sales_target.alias("target")
            .merge(
                invalid_snapshot.select("sale_id").alias("source"),
                "target.sale_id = source.sale_id"
            )
            .whenMatchedDelete()
            .execute()
        )

        # Remove newly valid sales from Quarantine
        quarantine_target = DeltaTable.forName(
            spark,
            "bronze_sales_quarantine"
        )

        (
            quarantine_target.alias("target")
            .merge(
                valid_snapshot.select("sale_id").alias("source"),
                "target.sale_id = source.sale_id"
            )
            .whenMatchedDelete()
            .execute()
        )

        print("Sales reconciliation completed")
        print("Current Valid Source Sales:", valid_count)
        print("Current Invalid Source Sales:", invalid_count)

    finally:
        valid_snapshot.unpersist()
        invalid_snapshot.unpersist()

else:

    print("Sales source unchanged — skipping reconciliation")

# Validate existing target tables
active_sales = spark.table("bronze_sales_incremental")
quarantined_sales = spark.table("bronze_sales_quarantine")

active_count = active_sales.count()
quarantine_count = quarantined_sales.count()

overlap_count = (
    active_sales.select("sale_id")
    .intersect(quarantined_sales.select("sale_id"))
    .count()
)

print("Active Sales:", active_count)
print("Active Quarantine:", quarantine_count)
print("Overlapping Sale IDs:", overlap_count)

if overlap_count != 0:
    raise ValueError(
        "Sales exist in both active and quarantine tables"
    )


In [ ]:

# STEP 81: Final Production Bronze Validation
# Supports soft-deleted Customers and Products
# Replaces original Step 62

from pyspark.sql import functions as F

customers = spark.table("bronze_customers_incremental")
products = spark.table("bronze_products_incremental")
sales = spark.table("bronze_sales_incremental")
quarantine = spark.table("bronze_sales_quarantine")

tables = {
    "Customers": (customers, "customer_id"),
    "Products": (products, "product_id"),
    "Sales": (sales, "sale_id"),
    "Quarantine": (quarantine, "sale_id")
}

# 1. Validate primary keys
for name, (df, key) in tables.items():

    total = df.count()
    unique = df.select(key).distinct().count()
    nulls = df.filter(F.col(key).isNull()).count()

    print(
        f"{name}: Total={total}, "
        f"Duplicates={total - unique}, "
        f"Null IDs={nulls}"
    )

    if total != unique or nulls > 0:
        raise ValueError(
            f"{name} failed primary key validation"
        )

# 2. Validate soft-delete metadata
for name, df in [
    ("Customers", customers),
    ("Products", products)
]:

    invalid_status = df.filter(
        F.col("is_deleted").isNull()
    ).count()

    invalid_timestamp = df.filter(
        (F.col("is_deleted") == True)
        & F.col("deleted_at").isNull()
    ).count()

    active_with_timestamp = df.filter(
        (F.col("is_deleted") == False)
        & F.col("deleted_at").isNotNull()
    ).count()

    active_count = df.filter(
        F.col("is_deleted") == False
    ).count()

    deleted_count = df.filter(
        F.col("is_deleted") == True
    ).count()

    print(
        f"{name}: Active={active_count}, "
        f"Soft Deleted={deleted_count}"
    )

    if (
        invalid_status > 0
        or invalid_timestamp > 0
        or active_with_timestamp > 0
    ):
        raise ValueError(
            f"{name} has invalid soft-delete metadata"
        )

# 3. Only active dimensions are valid references
active_customers = customers.filter(
    F.col("is_deleted") == False
).select("customer_id")

active_products = products.filter(
    F.col("is_deleted") == False
).select("product_id")

invalid_customer_sales = sales.join(
    active_customers,
    "customer_id",
    "left_anti"
).count()

invalid_product_sales = sales.join(
    active_products,
    "product_id",
    "left_anti"
).count()

# 4. Validate Sales / Quarantine overlap
overlapping_sales = (
    sales.select("sale_id")
    .intersect(quarantine.select("sale_id"))
    .count()
)

print(
    "Active Sales with missing/inactive customers:",
    invalid_customer_sales
)

print(
    "Active Sales with missing/inactive products:",
    invalid_product_sales
)

print(
    "Sales in both active and quarantine:",
    overlapping_sales
)

if invalid_customer_sales != 0:
    raise ValueError(
        "Active Sales contain missing or inactive customers"
    )

if invalid_product_sales != 0:
    raise ValueError(
        "Active Sales contain missing or inactive products"
    )

if overlapping_sales != 0:
    raise ValueError(
        "Sales and Quarantine overlap"
    )

# 5. Reconcile source Sales when processed
if should_process_sales:

    source_ids = incoming_sales_clean.select("sale_id")

    processed_ids = (
        sales.select("sale_id")
        .union(quarantine.select("sale_id"))
        .distinct()
    )

    unprocessed = source_ids.join(
        processed_ids,
        "sale_id",
        "left_anti"
    ).count()

    print("Unprocessed Source Sales:", unprocessed)

    if unprocessed != 0:
        raise ValueError(
            "Some source sales were not processed"
        )

else:

    print(
        "Source Sales reconciliation check skipped "
        "(unchanged source)"
    )

print("FINAL PRODUCTION BRONZE VALIDATION PASSED")


In [ ]:

# STEP 63: Conditional Source Fingerprint Registration

from pyspark.sql import functions as F
from delta.tables import DeltaTable

control_table = "ctl_source_file_hashes"

# Identify sources that required processing
processed_sources = {
    "customers": should_process_customers,
    "products": should_process_products,
    "sales": should_process_sales
}

sources_to_register = [
    name for name, processed in processed_sources.items()
    if processed
]

if sources_to_register:

    # Verify source files have not changed since hashing
    for source_name in sources_to_register:

        latest_hash = calculate_file_hash(
            source_paths[source_name]
        )

        if latest_hash != source_hashes[source_name]:
            raise ValueError(
                f"{source_name} changed during processing. "
                "Fingerprint registration aborted."
            )

    # Build fingerprint records
    hash_records = [
        (name, source_hashes[name])
        for name in sources_to_register
    ]

    hash_df = (
        spark.createDataFrame(
            hash_records,
            ["source_name", "file_hash"]
        )
        .withColumn("processed_at", F.current_timestamp())
    )

    control_delta = DeltaTable.forName(
        spark,
        control_table
    )

    (
        control_delta.alias("target")
        .merge(
            hash_df.alias("source"),
            "target.source_name = source.source_name"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        "Fingerprints registered for:",
        ", ".join(sources_to_register)
    )

else:

    print("No source processing required — fingerprints unchanged")

print(
    "Registered Sources:",
    spark.table(control_table).count()
)
